In [18]:
#Importiamo le librerie necessarie per lo svolgimento dell'esercizio
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [19]:
#1Carico il file PDF
reader = PdfReader("data/carta_servizi_poliambulatorio.pdf") 
docs = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    docs.append(Document(page_content=text, metadata={"sezione": f"Pagina {i+1}"}))
#2Chunking 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True
)
chunks = text_splitter.split_documents(docs)
print(f"Creati {len(chunks)} chunk dal documento.")
#3Inizializziamo gli embedding con il modello scelto ovvero "bge-m3"
embeddings = OllamaEmbeddings(model="bge-m3")
#infine creiamo e popoliamo il database vettoriale con chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("Database vettoriale inizializzato con successo!")

Creati 5 chunk dal documento.
Database vettoriale inizializzato con successo!


In [26]:
k = 3
embeddings = OllamaEmbeddings(model="bge-m3")
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
llm = ChatOllama(model="qwen2.5",temperature=0)
system_prompt = (
    "Sei l'assistente virtuale informativo del Poliambulatorio Salus Levante. "
    "Rispondi alla domanda del paziente utilizzando **unicamente** i contesti estratti dalla Carta dei Servizi qui sotto.\n"
    "Se l'informazione non è presente nel documento, di' chiaramente che non la trovi, "
    "senza inventare orari, tariffe o regole (evita assolutamente allucinazioni).\n"
    "Cita sempre la sezione o il riferimento del documento da cui prendi l'informazione.\n\n"
    "Contesto:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])
def formatta_docu(docs):
    return "\n\n".join(doc.page_content for doc in docs)
rag_chain = (
    {"context": retriever | formatta_docu, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
domanda_test = "Quali sono le regole per le disdette?"
test = rag_chain.invoke(domanda_test)
print(f"Domanda test: {domanda_test}")
print(f"Risposta test: {test}")

Domanda test: Quali sono le regole per le disdette?
Risposta test: Le visite possono essere disdette gratuitamente fino a 48 ore prima. Per le disdette tardive o la non comparezione, ci sono tariffe da pagare. Questa informazione si trova nella sezione "Disdette" del documento.


In [ ]:
def nodo_rag(state):
    domanda = state.get("input")
    risposta = rag_chain.invoke(domanda)
    return {"response":risposta}
